# 📚 Guía de referencia: Asimetría, Curtosis y Transformación Logarítmica
Este notebook explica paso a paso qué son estos conceptos y qué hizo el código de análisis de `precio_venta`.

---
## 1️⃣ ¿Qué es una distribución normal? (base para entender todo lo demás)

Imagina que mides la altura de 1000 personas.
- La mayoría mide entre 1.60m y 1.80m → están en el **centro**
- Muy poquitos miden 1.40m o 2.10m → están en los **extremos**

Eso es una **distribución normal**: una campana simétrica donde los datos se acumulan en el centro.

Los modelos de Machine Learning funcionan **mucho mejor** cuando los datos se parecen a esa campana.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Generamos datos que siguen una distribución normal (como alturas de personas)
datos_normales = np.random.normal(loc=170, scale=10, size=1000)  # media=170cm, desv=10cm

plt.figure(figsize=(8, 4))
plt.hist(datos_normales, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
plt.title('Ejemplo de distribución NORMAL (forma de campana)')
plt.xlabel('Altura (cm)')
plt.ylabel('Frecuencia')
plt.axvline(np.mean(datos_normales), color='red', linestyle='--', label='Media')
plt.legend()
plt.tight_layout()
plt.show()
# → Deberías ver una campana simétrica centrada en 170cm

---
## 2️⃣ ¿Qué es la ASIMETRÍA (Skewness)?

La asimetría mide **qué tan torcida está la campana** hacia un lado.

- **Skewness = 0** → campana perfectamente simétrica ✅
- **Skewness > 1** → la cola se va hacia la DERECHA (hay valores muy grandes jalando)
- **Skewness < -1** → la cola se va hacia la IZQUIERDA

### En nuestros datos de precio_venta:
- `skewness = 52.47` → asimetría EXTREMA hacia la derecha
- Significa: la mayoría de propiedades cuestan ~$700M, pero hay algunas que cuestan $4,250,000M
- Esos precios gigantes "jalan" la campana hacia la derecha de forma brutal

In [ ]:
# Simulamos datos con asimetría alta (como precios de propiedades)
# np.random.exponential genera datos donde la mayoría son pequeños pero hay algunos MUY grandes
datos_asimetricos = np.random.exponential(scale=700, size=1000)  # simulando precios en millones

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Gráfico izquierdo: distribución normal (lo que queremos)
axes[0].hist(datos_normales, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Distribución NORMAL\nSkewness ≈ {np.array(datos_normales).tolist()}')
axes[0].set_title('Distribución NORMAL\nSkewness ≈ 0 ✅')
axes[0].set_xlabel('Valor')
axes[0].set_ylabel('Frecuencia')

# Gráfico derecho: distribución asimétrica (nuestro problema)
axes[1].hist(datos_asimetricos, bins=40, color='tomato', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Distribución ASIMÉTRICA\nSkewness ≈ {round(float(np.array(datos_asimetricos).__class__(0)), 2)} (cola larga a la derecha) 🚨')
axes[1].set_title('Distribución ASIMÉTRICA\nCola larga a la derecha 🚨')
axes[1].set_xlabel('Valor (ej: precio en millones)')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()

# Calculamos la asimetría de ambas
from scipy import stats
print(f'Skewness datos normales:    {stats.skew(datos_normales):.4f}  → cerca de 0, bien ✅')
print(f'Skewness datos asimétricos: {stats.skew(datos_asimetricos):.4f}  → alto, problema 🚨')
print(f'Skewness precio_venta real: 52.4725  → extremadamente alto 🚨🚨')

---
## 3️⃣ ¿Qué es la CURTOSIS?

La curtosis mide **qué tan puntiaguda o achatada** es la campana, y qué tan largas son las colas.

- **Curtosis = 0** → campana normal estándar
- **Curtosis muy alta (ej: 3049)** → la campana es MUY puntiaguda y tiene colas MUY largas
  - Significa: hay valores extremos muy, muy alejados del centro

### En nuestros datos:
- `Curtosis = 3049.96` → hay outliers BRUTALMENTE alejados
- El precio máximo ($4,250,000M) está tan lejos de la mediana ($700M) que distorsiona todo

In [ ]:
# Visualizando el concepto de curtosis
x = np.linspace(-5, 5, 300)

# Distribución normal estándar
from scipy.stats import norm, laplace
y_normal = norm.pdf(x, 0, 1)
y_puntiaguda = laplace.pdf(x, 0, 0.7)  # más puntiaguda = curtosis alta

plt.figure(figsize=(8, 4))
plt.plot(x, y_normal, 'steelblue', linewidth=2, label='Curtosis normal (colas moderadas)')
plt.plot(x, y_puntiaguda, 'tomato', linewidth=2, label='Curtosis alta (colas largas = más outliers)')
plt.title('Curtosis: cómo afecta la forma de la distribución')
plt.xlabel('Valor')
plt.ylabel('Densidad')
plt.legend()
plt.tight_layout()
plt.show()
# Las colas largas = hay valores MUY extremos (como esa propiedad de $4,250,000M)

---
## 4️⃣ ¿Qué es el LOGARITMO (log)?

El logaritmo es una operación matemática que **comprime los números grandes**.

### Ejemplo simple:
```
log(1,000)       =  6.9
log(1,000,000)   = 13.8   ← 1000x más grande, pero solo el doble en log
log(1,000,000,000) = 20.7  ← 1000x más grande de nuevo, solo +7 en log
```

### ¿Por qué esto ayuda con los precios?
- Precio mínimo: $1,000,000 → log = 13.8
- Precio mediana: $700,000,000 → log = 20.4
- Precio máximo: $4,250,000,000,000 → log = 29.1

En escala original la diferencia es **4,249,999M**. En escala log es solo **15.3**.
→ Los outliers ya no dominan tanto, la distribución se aplana y se equilibra.

In [ ]:
# Demostración del efecto del logaritmo
precios_ejemplo = np.array([1_000_000, 100_000_000, 700_000_000, 1_300_000_000, 4_250_000_000_000])
nombres = ['$1M\n(mínimo)', '$100M', '$700M\n(mediana)', '$1,300M\n(percentil 75)', '$4,250,000M\n(máximo)']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Escala original: los valores pequeños casi no se ven
axes[0].bar(range(5), precios_ejemplo / 1e6, color='tomato', alpha=0.8)
axes[0].set_xticks(range(5))
axes[0].set_xticklabels(nombres, fontsize=8)
axes[0].set_title('Escala ORIGINAL\nEl máximo aplasta a todos los demás 🚨')
axes[0].set_ylabel('Millones COP')
# → La barra de $4,250,000M hace que las demás no se vean

# Escala logarítmica: todos los valores son comparables
axes[1].bar(range(5), np.log(precios_ejemplo), color='seagreen', alpha=0.8)
axes[1].set_xticks(range(5))
axes[1].set_xticklabels(nombres, fontsize=8)
axes[1].set_title('Escala LOGARÍTMICA\nAhora todos los valores son comparables ✅')
axes[1].set_ylabel('log(precio)')

# Añadimos los valores encima de cada barra
for i, v in enumerate(np.log(precios_ejemplo)):
    axes[1].text(i, v + 0.1, f'{v:.1f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

print('Diferencia en escala original:', f'{(precios_ejemplo[-1] - precios_ejemplo[0])/1e6:,.0f} millones COP')
print('Diferencia en escala log:     ', f'{np.log(precios_ejemplo[-1]) - np.log(precios_ejemplo[0]):.1f} unidades')

---
## 5️⃣ ¿Qué hizo el código original paso a paso?

```python
precio = df_sel['precio_venta'].dropna()
```
→ Tomó la columna de precios y descartó filas vacías (aunque no había ninguna)

```python
precio.describe()
```
→ Calculó count, media, desviación estándar, mínimo, percentiles y máximo

```python
precio.skew()       # = 52.47
precio.kurtosis()   # = 3049.96
```
→ Midió qué tan torcida y puntiaguda es la distribución → ambas muy altas = problema

```python
np.log(precio).skew()   # = 0.10
```
→ Aplicó logaritmo a todos los precios y midió la asimetría → ¡bajó de 52 a 0.10!

```python
# Gráfico 1: histograma original
axes[0].hist(precio / 1e6, bins=60, ...)
```
→ Mostró cómo se ven los precios en su escala real (en millones COP)

```python
# Gráfico 2: histograma log-transformado  
axes[1].hist(np.log(precio), bins=60, ...)
```
→ Mostró que al aplicar log(), la distribución se parece a una campana normal

```python
# Gráfico 3: boxplot
axes[2].boxplot(precio / 1e6, ...)
```
→ Mostró la caja con bigotes donde se ven claramente los puntos outliers (los precios extremos)

In [ ]:
# Reproducción simplificada del análisis original con comentarios detallados
# (usando datos simulados similares a precio_venta)

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Simulamos precios similares a los reales (distribución log-normal)
# En tu caso real usarías: precio = df_sel['precio_venta'].dropna()
np.random.seed(42)
precio_simulado = np.random.lognormal(mean=20.4, sigma=1.2, size=27270)  # similar a precio_venta real

# ── PASO 1: Ver estadísticas básicas ──────────────────────────────────────────
print('=== ESTADÍSTICAS DESCRIPTIVAS ===')
print(f'Media:   {precio_simulado.mean():>20,.0f}  ← inflada por outliers')
print(f'Mediana: {np.median(precio_simulado):>20,.0f}  ← precio típico REAL')
print(f'Mínimo:  {precio_simulado.min():>20,.0f}')
print(f'Máximo:  {precio_simulado.max():>20,.0f}')
print()

# ── PASO 2: Medir la asimetría ─────────────────────────────────────────────── 
# skew=0 → simétrico, skew>1 → cola derecha larga, skew<-1 → cola izquierda larga
skewness_original = stats.skew(precio_simulado)
print(f'=== ASIMETRÍA ===')
print(f'Skewness original: {skewness_original:.4f}  → muy alto, distribución torcida 🚨')
print()

# ── PASO 3: Aplicar logaritmo y medir de nuevo ────────────────────────────────
# np.log() calcula el logaritmo natural de cada precio
# Esto comprime los valores grandes y expande los pequeños
precio_log = np.log(precio_simulado)
skewness_log = stats.skew(precio_log)
print(f'=== EFECTO DEL LOGARITMO ===')
print(f'Skewness con log(): {skewness_log:.4f}  → cercano a 0, distribución casi normal ✅')
print(f'Mejora: de {skewness_original:.1f} → {skewness_log:.4f}')
print()

# ── PASO 4: Graficar los 3 paneles ────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Análisis completo de precio_venta', fontsize=14, fontweight='bold')

# Panel 1: distribución original
# Dividimos entre 1e6 para mostrar en millones (más legible)
axes[0].hist(precio_simulado / 1e6, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Distribución Original\nSkewness = {skewness_original:.1f} 🚨')
axes[0].set_xlabel('Precio (millones COP)')
axes[0].set_ylabel('Frecuencia')
# → Verás una barra gigante a la izquierda y una cola muy larga a la derecha

# Panel 2: distribución después de aplicar log()
# Aquí ya no se ven los valores reales sino log(precio), pero la FORMA es lo importante
axes[1].hist(precio_log, bins=60, color='seagreen', edgecolor='white', alpha=0.8)
axes[1].set_title(f'Distribución Log-transformada\nSkewness = {skewness_log:.4f} ✅')
axes[1].set_xlabel('log(Precio)  ← unidad abstracta')
axes[1].set_ylabel('Frecuencia')
# → Ahora sí parece una campana simétrica → el modelo aprenderá mucho mejor

# Panel 3: boxplot para ver outliers visualmente
# La caja = 50% central de los datos
# Los bigotes = rango esperado
# Los puntos fuera de los bigotes = OUTLIERS
axes[2].boxplot(precio_simulado / 1e6, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].set_title('Boxplot — Outliers visibles')
axes[2].set_ylabel('Precio (millones COP)')
# → Verás la caja abajo (donde están la mayoría de precios)
# → y puntos sueltos arriba (los precios extremos)

plt.tight_layout()
plt.show()

---
## 6️⃣ Conclusión: ¿Qué hacer con esto en el modelo?

### El flujo correcto es:

```
DATOS ORIGINALES          ENTRENAR MODELO          PREDICCIÓN FINAL
precio = $700,000,000  →  log(700M) = 20.4  →  modelo aprende  →  predice 20.6  →  exp(20.6) = $889M
```

1. **Antes de entrenar**: aplicar `np.log(precio_venta)` a la columna objetivo
2. **El modelo aprende** sobre los valores log (más manejables)
3. **Al predecir**: el modelo da un valor log (ej: 20.6)
4. **Convertir de vuelta**: aplicar `np.exp(20.6)` → obtienes el precio en COP real

### ¿Por qué no entrenar directamente sobre el precio?
- El modelo vería que $4,250,000M es un valor 'correcto'
- Intentaría predecir bien ese outlier extremo
- Para hacerlo, sacrificaría precisión en el rango típico ($400M–$1,300M)
- Resultado: pésimas predicciones para el 90% de las propiedades normales

In [ ]:
# Demostración final: cómo aplicar la transformación en tu pipeline real

# PASO 1: Crear la columna transformada
# df_sel['log_precio_venta'] = np.log(df_sel['precio_venta'])

# PASO 2: Usar log_precio_venta como variable objetivo al entrenar
# y = df_sel['log_precio_venta']  # ← esto va al modelo

# PASO 3: Cuando el modelo predice, convertir de vuelta
# prediccion_log = modelo.predict(X_test)      # el modelo da log(precio)
# prediccion_real = np.exp(prediccion_log)     # esto da el precio en COP

# Ejemplo numérico para que quede claro:
precio_real = 700_000_000  # $700 millones COP
precio_en_log = np.log(precio_real)
precio_recuperado = np.exp(precio_en_log)

print(f'Precio original:          {precio_real:>15,.0f} COP')
print(f'Después de np.log():      {precio_en_log:>15.4f}       ← esto aprende el modelo')
print(f'Después de np.exp():      {precio_recuperado:>15,.0f} COP  ← precio real recuperado ✅')
print()
print('log() y exp() son operaciones INVERSAS: lo que una hace, la otra lo deshace.')